In [ ]:
import warnings; warnings.filterwarnings("ignore")
import torch, gc, os, json, re, time, random, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
import accelerate

random.seed(42); torch.manual_seed(42)
print(f"CUDA: {torch.version.cuda}  |  GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"
print("HF token set")

# GPU capability check (P100 sm_60 incompatible with PyTorch 2.13)
sm = torch.cuda.get_device_capability(0)
print(f"SM: {sm}")

USE_4BIT = False
BNB_CONFIG = None
if sm[0] >= 7:
    print("GPU compatible - attempting 4-bit...")
    try:
        os.system("pip install -q bitsandbytes --no-cache-dir 2>/dev/null")
        import bitsandbytes
        from transformers import BitsAndBytesConfig
        BNB_CONFIG = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        USE_4BIT = True
        print(f"bitsandbytes {bitsandbytes.__version__} - 4-bit ready")
    except Exception as e:
        print(f"4-bit unavailable: {e}")
else:
    print("GPU too old (sm<70). Skipping 4-bit. Using fp16 offload.")

print(f"PyTorch: {torch.__version__}  |  Accelerate: {accelerate.__version__}")


In [ ]:
ITEMS = [
    {"instr":"Explain how photosynthesis works.","resp":"Plants use sunlight to convert CO2 and water into glucose and oxygen."},
    {"instr":"What is the theory of relativity?","resp":"Einstein theory says space and time are relative."},
    {"instr":"Describe the water cycle.","resp":"Water evaporates, forms clouds, returns as precipitation."},
    {"instr":"What causes earthquakes?","resp":"Tectonic plates shift, releasing seismic waves."},
    {"instr":"Explain how vaccines work.","resp":"Vaccines train immune system to recognize pathogens."},
    {"instr":"What is DNA?","resp":"DNA carries genetic instructions for growth and reproduction."},
    {"instr":"Describe the solar system.","resp":"8 planets orbiting the Sun."},
    {"instr":"What is entropy?","resp":"Disorder measure. Always increases per 2nd law."},
    {"instr":"How do batteries work?","resp":"Chemical reactions create electron flow via electrolyte."},
    {"instr":"What is a black hole?","resp":"Gravity so strong light cannot escape."},
    {"instr":"What is machine learning?","resp":"AI learning patterns from data without explicitly programming."},
    {"instr":"Describe cloud computing.","resp":"On-demand computing resources over the internet."},
    {"instr":"What is an API?","resp":"Interface for different software to communicate with each other."},
    {"instr":"Explain encryption.","resp":"Algorithms that encode data using cryptographic keys."},
    {"instr":"What is a database index?","resp":"Structure speeding up data retrieval like a book index."},
    {"instr":"What is Python?","resp":"High-level readable interpreted programming language."},
    {"instr":"Explain the internet.","resp":"Global network of computers communicating via TCP/IP."},
    {"instr":"What is blockchain?","resp":"Distributed ledger recording transactions in immutable blocks."},
    {"instr":"What is an operating system?","resp":"Software managing hardware resources for applications."},
    {"instr":"Explain neural networks.","resp":"Computational systems inspired by biological neural networks."},
    {"instr":"What caused World War I?","resp":"Assassination of Archduke Franz Ferdinand triggered alliance system."},
    {"instr":"Explain democracy.","resp":"System where citizens vote for representatives."},
    {"instr":"What was the Renaissance?","resp":"14th-17th century cultural rebirth in Europe."},
    {"instr":"Describe capitalism.","resp":"Economic system with private ownership and profit motive."},
    {"instr":"What is the United Nations?","resp":"International organization for peace and cooperation."},
    {"instr":"Explain the Cold War.","resp":"Geopolitical tension between US and USSR 1947-1991."},
    {"instr":"What is ethics?","resp":"Branch of philosophy studying moral right and wrong."},
    {"instr":"Describe the Enlightenment.","resp":"18th century intellectual movement emphasizing reason."},
    {"instr":"What is the Constitution?","resp":"Supreme law establishing government structure and rights."},
    {"instr":"Explain globalization.","resp":"Increasing interconnectedness of world economies and cultures."},
    {"instr":"How to make a budget?","resp":"Track income and expenses, allocate for needs and savings."},
    {"instr":"What is healthy eating?","resp":"Balanced diet with fruits, vegetables, protein, whole grains."},
    {"instr":"Explain first aid for burns.","resp":"Cool with running water, cover with sterile dressing."},
    {"instr":"What is recycling?","resp":"Converting waste materials into new reusable products."},
    {"instr":"How does a car engine work?","resp":"Fuel combustion in cylinders drives pistons turning crankshaft."},
    {"instr":"What is renewable energy?","resp":"Energy from replenishable sources like sun and wind."},
    {"instr":"How to grow vegetables?","resp":"Plant seeds in soil with water, sunlight, and nutrients."},
    {"instr":"What is a mortgage?","resp":"Loan for purchasing property paid in monthly installments."},
    {"instr":"Explain the stock market.","resp":"Platform where company shares are bought and sold."},
    {"instr":"What is inflation?","resp":"General increase in prices reducing purchasing power."},
    {"instr":"What is the Pythagorean theorem?","resp":"a squared plus b squared equals c squared."},
    {"instr":"Explain a prime number.","resp":"Number divisible only by 1 and itself."},
    {"instr":"What is calculus?","resp":"Mathematics of change using derivatives and integrals."},
    {"instr":"What is a logarithm?","resp":"Inverse of exponentiation, log base 10 of 100 is 2."},
    {"instr":"What is the quadratic formula?","resp":"x equals negative b plus or minus square root of b squared minus 4ac over 2a."},
    {"instr":"Explain probability.","resp":"Likelihood of events occurring, from 0 to 1."},
    {"instr":"What is a proof?","resp":"Logical demonstration that a statement follows from axioms."},
    {"instr":"What is a function?","resp":"Relation where each input maps to exactly one output."},
    {"instr":"Explain standard deviation.","resp":"Measure of dispersion from the mean."},
    {"instr":"What is a matrix?","resp":"Rectangular array of numbers for linear transformations."},
]
print(f"{len(ITEMS)} items")


In [ ]:
PROBE_VARIANTS = {"rubric_order": ["control", "reversed", "random"], "score_id": ["numeric", "letter", "descriptive"], "reference_answer": ["none", "good", "poor"]}

def make_prompt(item, pt, var):
    i, r = item["instr"], item["resp"]
    if pt == "rubric_order":
        rub = {"control":"Score from 1-5 (where 1 is worst, 5 is best)", "reversed":"Score from 1-5 (where 1 is best, 5 is worst)", "random":"Score from 1-5"}[var]
        return f"Evaluate the following response.\n### Instruction: {i}\n### Response: {r}\n### {rub}\n### Score:"
    elif pt == "score_id":
        rub = {"numeric":"Score from 1-5", "letter":"Score A-E (where A is best, E is worst)", "descriptive":"Rate: Poor, Fair, Good, Very Good, Excellent"}[var]
        return f"Evaluate the following response.\n### Instruction: {i}\n### Response: {r}\n### {rub}\n### Score:"
    elif pt == "reference_answer":
        ref = "" if var == "none" else "\n### Excellent response: This is a perfect answer." if var == "good" else "\n### Poor response: This is wrong."
        return f"Evaluate the following response.\n### Instruction: {i}{ref}\n### Response: {r}\n### Score from 1-5 (where 1 is worst, 5 is best)\n### Score:"
print("Probes ready")

test_prompt = make_prompt(ITEMS[0], "rubric_order", "control")
print(f"Sample prompt (first 200 chars): {repr(test_prompt[:200])}")


In [ ]:
LETTER_MAP = {"A":5,"B":4,"C":3,"D":2,"E":1,"F":1}
DESCRIPTIVE_MAP = {"excellent":5,"very good":4,"good":3,"fair":2,"poor":1,"great":5,"average":3,"bad":1,"terrible":1,"perfect":5,"outstanding":5,"exceptional":5,"satisfactory":3,"inadequate":1,"mediocre":2}

def parse_score(text, variant):
    if not text: return None
    t = text.strip().lower()
    nums = re.findall(r"\b([1-5])\b", t)
    let = re.search(r"\b([a-f])\b", t)
    desc = None
    for w,s in sorted(DESCRIPTIVE_MAP.items(), key=lambda x:-len(x[0])):
        if w in t: desc = float(s); break
    if variant == "numeric":
        if nums: return float(nums[0])
        if let: return float(LETTER_MAP.get(let.group(1).upper(),0))
        return desc
    elif variant == "letter":
        if let: return float(LETTER_MAP.get(let.group(1).upper(),0))
        if nums: return float(nums[0])
        return desc
    elif variant == "descriptive":
        if desc: return desc
        if nums: return float(nums[0])
        if let: return float(LETTER_MAP.get(let.group(1).upper(),0))
        return None
    else:
        if nums: return float(nums[0])
        if let: return float(LETTER_MAP.get(let.group(1).upper(),0))
        return desc

def score_model(m, tok, p, var, dev):
    inp = tok(p, return_tensors="pt", truncation=True, max_length=1024).to(dev)
    with torch.no_grad():
        o = m.generate(**inp, max_new_tokens=20, temperature=0.0, do_sample=False, pad_token_id=tok.eos_token_id)
    f = tok.decode(o[0], skip_special_tokens=True)
    g = f[len(p):].strip()
    if "### Score:" in g:
        s = parse_score(g.split("### Score:")[-1].split("###")[0].strip(), var)
        if s and 1<=s<=5: return s
    s = parse_score(g, var)
    if s and 1<=s<=5: return s
    x = re.findall(r"### Score:?\s*([1-5])", f)
    if x: return float(x[-1])
    return None
print("Parser ready")


In [ ]:
MODELS_7B = [
    ("mistralai/Mistral-7B-v0.3", "mistralai/Mistral-7B-Instruct-v0.3", "Mistral-7B-v0.3"),
    ("deepseek-ai/deepseek-llm-7b-base", "deepseek-ai/deepseek-llm-7b-chat", "DeepSeek-7B"),
    ("google/gemma-2-9b", "google/gemma-2-9b-it", "Gemma2-9B"),
    ("allenai/OLMo-7B-hf", "allenai/OLMo-7B-0724-Instruct-hf", "OLMo-7B"),
]
MODELS_SMALL = [
    ("Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-0.5B-Instruct", "Qwen2.5-0.5B"),
    ("Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-3B-Instruct", "Qwen2.5-3B"),
]
DEEPSEEK_IDS = {"deepseek-ai/deepseek-llm-7b-chat"}
print(f"7B/9B: {len(MODELS_7B)} families | Small: {len(MODELS_SMALL)} families")
print("Note: Gemma2-9B requires license at https://huggingface.co/google/gemma-2-9b")


In [ ]:
CHECKPOINT = "final_results.json"
all_results = {}
if os.path.exists(CHECKPOINT):
    try:
        with open(CHECKPOINT) as f: all_results = json.load(f)
        print(f"Resumed: {len(all_results)} models from checkpoint")
    except: all_results = {}

device = torch.device("cuda")

for fams, big in [(MODELS_7B, True), (MODELS_SMALL, False)]:
    for bid, iid, name in fams:
        for mid, mn in [(bid, name)] + ([(iid, f"{name}-IT")] if iid else []):
            if mn in all_results and all(v is not None for v in all_results[mn].values()):
                print(f"Skipping {mn} (done)"); continue
            print(f"\n{'='*60}\n{mn}  ({mid})\n{'='*60}")
            model = None
            try:
                kw = {"quantization_config": BNB_CONFIG, "device_map": "auto"} if (big and USE_4BIT) else \
                     {"torch_dtype": torch.float16, "device_map": "auto", "max_memory": {0:"14.5GiB", "cpu":"32GiB"}, "offload_folder": "off"} if big else \
                     {"torch_dtype": torch.float16, "device_map": "auto"}
                tok = AutoTokenizer.from_pretrained(mid, token=os.environ["HF_TOKEN"])
                if tok.pad_token is None: tok.pad_token = tok.eos_token
                model = AutoModelForCausalLM.from_pretrained(mid, **kw, token=os.environ["HF_TOKEN"])
                model.eval()
                print(f"Loaded: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")
            except Exception as e:
                print(f"FAIL: {e}"); continue
            ct = mid in DEEPSEEK_IDS; sc = {pt:{} for pt in PROBE_VARIANTS}
            for pt, vv in PROBE_VARIANTS.items():
                for pv in vv:
                    its = []
                    for idx, it in enumerate(ITEMS):
                        p = make_prompt(it, pt, pv)
                        if ct:
                            p = tok.apply_chat_template([{"role":"user","content":p}], tokenize=False, add_generation_prompt=True)
                        reps = []
                        for _ in range(3):
                            try:
                                s = score_model(model, tok, p, pv, device)
                                if s and 1<=s<=5: reps.append(s)
                            except: pass
                        if reps: its.append(np.mean(reps))
                        if (idx+1)%25==0: print(f"  [{pt}/{pv}] {idx+1}/{len(ITEMS)} ({len(its)} scored)", flush=True)
                    sc[pt][pv] = {"mean": float(np.mean(its)) if its else None}
                    if not its:
                        p2 = make_prompt(ITEMS[0], pt, pv)
                        if ct:
                            p2 = tok.apply_chat_template([{"role":"user","content":p2}], tokenize=False, add_generation_prompt=True)
                        inp = tok(p2, return_tensors="pt", truncation=True, max_length=1024).to(device)
                        with torch.no_grad():
                            o = model.generate(**inp, max_new_tokens=25, temperature=0.0, do_sample=False, pad_token_id=tok.eos_token_id)
                        d = tok.decode(o[0], skip_special_tokens=True)
                        print(f"  ZERO: \"{d[len(p2):].strip()[:150]}\")
            r = {}
            for pt, vv in PROBE_VARIANTS.items():
                ms = [sc[pt][v]["mean"] for v in vv if sc[pt][v]["mean"] is not None]
                r[pt] = round(max(ms)-min(ms),2) if len(ms)>=2 else None
            all_results[mn] = r
            print(f"DONE: {r}")
            with open(CHECKPOINT, "w") as f: json.dump(all_results, f, indent=2)
            del model; gc.collect(); torch.cuda.empty_cache(); time.sleep(3)

with open(CHECKPOINT, "w") as f: json.dump(all_results, f, indent=2)
print(f"\nDONE: {len(all_results)} models")


In [ ]:
with open("final_results.json") as f: data = json.load(f)
print(f"=== {len(data)} MODELS ===\n")
for n,r in sorted(data.items()):
    b = [f"{k}={v}" for k,v in r.items() if v is None]
    c = [f"{k}={v}" for k,v in r.items() if v is not None]
    print(f"  {"\u26a0" if b else "\u2705"} {n:28s}  {"  ".join(c)}")
complete = sum(1 for v in data.values() if all(x is not None for x in v.values()))
print(f"\nComplete: {complete}/{len(data)}")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !cp final_results.json /content/drive/MyDrive/scoring_bias_results.json
    print("Backed up to Drive")
except: pass
from google.colab import files
print("Downloading..."); files.download("final_results.json")
print("\n=== RAW JSON ===\n", json.dumps(data, indent=2))
